# CUDA 101: The CUDA Kernel

In [366]:
%%bash
nvidia-smi

Thu Oct 15 03:34:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 610.57.01              KMD Version: 610.88        CUDA UMD Version: 13.3     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3090        On  |   00000000:0C:00.0 Off |                  N/A |
| 64%   48C    P8             25W /  350W |    1700MiB /  24576MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [367]:
%%bash
for file in *.cpp *.hpp *.cu *.cuh *.so *.py threads __pycache__; do
    rm -rf "$file"
done

In [368]:
%%writefile parse_num.hpp
#pragma once
#include <cerrno>
#include <charconv>
#include <string_view>
#include <system_error>

template <typename T>
int parse_num(T& val, const std::string_view arg) {
  auto arg_end = arg.data() + arg.size();
  auto [read_end, read_err] = std::from_chars(arg.data(), arg_end, val);
  bool parse_err = read_err != std::errc{};
  bool partial_read = read_end != arg_end;
  if (parse_err) {
    return static_cast<int>(read_err);
  }
  if (partial_read) {
    return EINVAL;
  }
  return 0;
}


Writing parse_num.hpp


In [369]:
%%writefile check_grid.cuh
#pragma once
#include <cerrno>
#include <cuda_runtime.h>

inline int check_grid(const dim3& grid_shape, const cudaDeviceProp& prop) {
  if (grid_shape.x == 0 || grid_shape.y == 0 || grid_shape.z == 0) {
    return EINVAL;
  }
  if (grid_shape.x > prop.maxGridSize[0] || grid_shape.y > prop.maxGridSize[1] || grid_shape.z > prop.maxGridSize[2]) {
    return ERANGE;
  }
  return 0;
}

Writing check_grid.cuh


In [370]:
%%writefile check_block.cuh
#pragma once
#include <cerrno>
#include <cuda_runtime.h>

inline int check_block(const dim3& block_shape, const cudaDeviceProp& prop) {
  if (block_shape.x == 0 || block_shape.y == 0 || block_shape.z == 0) {
    return EINVAL;
  }
  if (block_shape.x > prop.maxThreadsDim[0] || block_shape.y > prop.maxThreadsDim[1] || block_shape.z > prop.maxThreadsDim[2]) {
    return ERANGE;
  }
  auto threads_per_block = block_shape.x * block_shape.y * block_shape.z;
  if (threads_per_block > prop.maxThreadsPerBlock) {
    return ERANGE;
  }
  return 0;
}

Writing check_block.cuh


In [371]:
%%writefile threads_welcome.cuh
#pragma once
#include <print>

__host__
inline void threads_welcome() {
  std::println("This program will launch a GPU kernel and print its thread number");
}

Writing threads_welcome.cuh


In [372]:
%%writefile get_thread.cuh
#pragma once
#include <cstdint>

__device__
inline uint32_t get_thread() {
  return blockIdx.x * blockDim.x + threadIdx.x;
}

Writing get_thread.cuh


In [373]:
%%writefile threads_kernel.cuh
#pragma once
#include <cstdint>
#include <cstdio>
#include "get_thread.cuh"

__global__
static void threads_kernel() {
  uint32_t thread = get_thread();
  printf("Hello from thread %u!\n", thread);
}

Writing threads_kernel.cuh


In [374]:
%%writefile run_kernel.cuh
#pragma once
#include <cuda_runtime.h>
#include "threads_kernel.cuh"

static inline cudaError_t run_kernel(const dim3& grid_shape, const dim3& block_shape) {
  threads_kernel<<<grid_shape, block_shape>>>();
  auto kernel_err = cudaGetLastError();
  if (kernel_err != cudaSuccess) {
    return kernel_err;
  }
  return cudaDeviceSynchronize();
}


Writing run_kernel.cuh


In [375]:
%%writefile threads.cu
#include <cstdint>
#include <cstdio>
#include <cstdlib>
#include <cstring>
#include <print>
#include <span>
#include <cuda_runtime.h>
#include "check_block.cuh"
#include "check_grid.cuh"
#include "parse_num.hpp"
#include "run_kernel.cuh"
#include "threads_welcome.cuh"

int main(int argc, char* argv[]) {
  std::span<char*> args(argv, argc);
  constexpr uint32_t device_id = 0;

  if (args.size() != 3) {
    std::println(stderr, "Usage: {} <grid size> <block size>", args[0]);
    std::exit(EXIT_FAILURE);
  }

  auto set_device_err = cudaSetDevice(device_id);
  if (set_device_err != cudaSuccess) {
    std::println(stderr, "Error setting device: {}", cudaGetErrorString(set_device_err));
    std::exit(EXIT_FAILURE);
  }

  cudaDeviceProp prop{};
  auto prop_err = cudaGetDeviceProperties(&prop, device_id);
  if (prop_err != cudaSuccess) {
    std::println(stderr, "Error getting device properties: {}", cudaGetErrorString(prop_err));
    std::exit(EXIT_FAILURE);
  }

  uint32_t grid_size{};
  auto grid_size_err = parse_num(grid_size, args[1]);
  if (grid_size_err != 0) {
    std::println(stderr, "Invalid grid size: {}", std::strerror(grid_size_err));
    std::exit(EXIT_FAILURE);
  }
  auto grid_shape = dim3{grid_size, 1, 1};
  auto grid_shape_err = check_grid(grid_shape, prop);
  if (grid_shape_err != 0) {
    std::println(stderr, "Invalid grid shape: {}", std::strerror(grid_shape_err));
    std::exit(EXIT_FAILURE);
  }

  uint32_t block_size{};
  auto block_size_err = parse_num(block_size, args[2]);
  if (block_size_err != 0) {
    std::println(stderr, "Invalid block size: {}", std::strerror(block_size_err));
    std::exit(EXIT_FAILURE);
  }
  auto block_shape = dim3{block_size, 1, 1};
  auto block_shape_err = check_block(block_shape, prop);
  if (block_shape_err != 0) {
    std::println(stderr, "Invalid block shape: {}", std::strerror(block_shape_err));
    std::exit(EXIT_FAILURE);
  }

  threads_welcome();
  auto kernel_err = run_kernel(grid_shape, block_shape);
  if (kernel_err != cudaSuccess) {
    std::println(stderr, "Kernel failed: {}", cudaGetErrorString(kernel_err));
    std::exit(EXIT_FAILURE);
  }

  return EXIT_SUCCESS;
}


Writing threads.cu


In [376]:
%%bash
nvcc -std=c++23 -arch=native -O3 "threads.cu" -o threads

In [377]:
%%bash
./threads 2 3

This program will launch a GPU kernel and print its thread number
Hello from thread 3!
Hello from thread 4!
Hello from thread 5!
Hello from thread 0!
Hello from thread 1!
Hello from thread 2!


# CUDA 102: Extending Python with C

In [378]:
%%writefile threads_lib.cu
#include <cstddef>
#include <cstdint>
#include <cuda_runtime.h>
#include "check_block.cuh"
#include "check_grid.cuh"
#include "run_kernel.cuh"
#include "threads_welcome.cuh"

extern "C" {
  cudaError_t c_set_device(uint32_t device_id) {
    return cudaSetDevice(device_id);
  }

  void c_welcome() {
    threads_welcome();
  }

  size_t c_prop_size() {
    return sizeof(cudaDeviceProp);
  }

  uint32_t c_uint32_max() {
    return UINT32_MAX;
  }

  cudaError_t c_get_prop(cudaDeviceProp* prop, uint32_t device_id) {
    return cudaGetDeviceProperties(prop, device_id);
  }

  int c_check_grid(const dim3* grid_shape, const cudaDeviceProp* prop) {
    return check_grid(*grid_shape, *prop);
  }

  int c_check_block(const dim3* block_shape, const cudaDeviceProp* prop) {
    return check_block(*block_shape, *prop);
  }

  cudaError_t c_run_kernel(const dim3* grid_shape, const dim3* block_shape) {
    return run_kernel(*grid_shape, *block_shape);
  }

  const char* c_get_error_string(cudaError_t err) {
    return cudaGetErrorString(err);
  }
}


Writing threads_lib.cu


In [379]:
%%bash
nvcc -std=c++23 -arch=native -O3 --compiler-options '-fPIC' -shared threads_lib.cu -o threads_lib.so

In [380]:
%%writefile threads.py
import os
import sys
from ctypes import (
    CDLL,
    POINTER,
    Structure,
    byref,
    c_byte,
    c_char_p,
    c_int,
    c_size_t,
    c_uint32,
)
from pathlib import Path

try:
    lib = CDLL(str(Path(__file__).parent / 'threads_lib.so'))
except OSError as e:
    raise RuntimeError("Failed to load threads_lib.so") from e

lib.c_prop_size.argtypes = []
lib.c_prop_size.restype  = c_size_t

lib.c_uint32_max.argtypes = []
lib.c_uint32_max.restype  = c_uint32

UINT32_MAX = lib.c_uint32_max()

class Dim3(Structure):
    _fields_ = [
        ("x", c_uint32),
        ("y", c_uint32),
        ("z", c_uint32)
    ]

    def __init__(self, x: int = 1, y: int = 1, z: int = 1):
        super().__init__(x, y, z)

    def __setattr__(self, name: str, value) -> None:
        if name in ("x", "y", "z"):
            if value < 0:
                raise ValueError(f"Invalid Dim3.{name}={value}: dimensions must be non-negative")
            if value > UINT32_MAX:
                raise ValueError(f"Invalid Dim3.{name}={value}: dimensions must not exceed {UINT32_MAX}")
        super().__setattr__(name, value)

    def __repr__(self) -> str:
        return f"Dim3({self.x}, {self.y}, {self.z})"

class DeviceProp(Structure):
    _fields_ = [
        ("data", c_byte * lib.c_prop_size())
    ]

lib.c_set_device.argtypes = [c_uint32]
lib.c_set_device.restype  = c_int

lib.c_welcome.argtypes = []
lib.c_welcome.restype  = None

lib.c_get_prop.argtypes = [POINTER(DeviceProp), c_uint32]
lib.c_get_prop.restype  = c_int

lib.c_check_grid.argtypes = [POINTER(Dim3), POINTER(DeviceProp)]
lib.c_check_grid.restype  = c_int

lib.c_check_block.argtypes = [POINTER(Dim3), POINTER(DeviceProp)]
lib.c_check_block.restype  = c_int

lib.c_run_kernel.argtypes = [POINTER(Dim3), POINTER(Dim3)]
lib.c_run_kernel.restype  = c_int

lib.c_get_error_string.argtypes = [c_int]
lib.c_get_error_string.restype  = c_char_p

def get_error_string(err: int) -> str:
    return lib.c_get_error_string(err).decode('utf-8')

def set_device(device_id: int) -> None:
    if (err := lib.c_set_device(device_id)) != 0:
        raise RuntimeError(f"Error setting device: {get_error_string(err)}")

def welcome() -> None:
    lib.c_welcome()

def get_prop(device_id: int) -> DeviceProp:
    prop = DeviceProp()
    if (err := lib.c_get_prop(byref(prop), device_id)) != 0:
        raise RuntimeError(f"Error getting device properties: {get_error_string(err)}")
    return prop

def check_grid(grid_shape: Dim3, prop: DeviceProp) -> None:
    if (err := lib.c_check_grid(byref(grid_shape), byref(prop))) != 0:
        raise ValueError(f"Invalid grid shape: {os.strerror(err)}")

def check_block(block_shape: Dim3, prop: DeviceProp) -> None:
    if (err := lib.c_check_block(byref(block_shape), byref(prop))) != 0:
        raise ValueError(f"Invalid block shape: {os.strerror(err)}")

def run_kernel(grid_shape: Dim3, block_shape: Dim3) -> None:
    if (err := lib.c_run_kernel(byref(grid_shape), byref(block_shape))) != 0:
        raise RuntimeError(f"Kernel failed: {get_error_string(err)}")

if __name__ == '__main__':
    device_id = 0

    if len(sys.argv) != 3:
        print(f"Usage: {sys.argv[0]} <grid size> <block size>", file=sys.stderr)
        sys.exit(1)

    set_device(device_id)
    prop = get_prop(device_id)

    grid_size = int(sys.argv[1])
    grid_shape = Dim3(grid_size, 1, 1)
    check_grid(grid_shape, prop)

    block_size = int(sys.argv[2])
    block_shape = Dim3(block_size, 1, 1)
    check_block(block_shape, prop)

    welcome()
    run_kernel(grid_shape, block_shape)


Writing threads.py


In [381]:
%%bash
python3 threads.py 2 3

This program will launch a GPU kernel and print its thread number
Hello from thread 3!
Hello from thread 4!
Hello from thread 5!
Hello from thread 0!
Hello from thread 1!
Hello from thread 2!


# CUDA 103: Hashing

In [382]:
%%writefile md5.cuh
#pragma once
#include <cstdint>
#include <cuda/std/bit>
#include <cuda/std/cstring>

constexpr uint32_t md5_digest_size = 16;

__device__
inline void md5(uint8_t* digest, const uint8_t* msg, uint32_t len) {
  const uint32_t s[64] = {
    7, 12, 17, 22,  7, 12, 17, 22,  7, 12, 17, 22,  7, 12, 17, 22,
    5,  9, 14, 20,  5,  9, 14, 20,  5,  9, 14, 20,  5,  9, 14, 20,
    4, 11, 16, 23,  4, 11, 16, 23,  4, 11, 16, 23,  4, 11, 16, 23,
    6, 10, 15, 21,  6, 10, 15, 21,  6, 10, 15, 21,  6, 10, 15, 21
  };
  const uint32_t K[64] = {
    0xd76aa478, 0xe8c7b756, 0x242070db, 0xc1bdceee,
    0xf57c0faf, 0x4787c62a, 0xa8304613, 0xfd469501,
    0x698098d8, 0x8b44f7af, 0xffff5bb1, 0x895cd7be,
    0x6b901122, 0xfd987193, 0xa679438e, 0x49b40821,
    0xf61e2562, 0xc040b340, 0x265e5a51, 0xe9b6c7aa,
    0xd62f105d, 0x02441453, 0xd8a1e681, 0xe7d3fbc8,
    0x21e1cde6, 0xc33707d6, 0xf4d50d87, 0x455a14ed,
    0xa9e3e905, 0xfcefa3f8, 0x676f02d9, 0x8d2a4c8a,
    0xfffa3942, 0x8771f681, 0x6d9d6122, 0xfde5380c,
    0xa4beea44, 0x4bdecfa9, 0xf6bb4b60, 0xbebfbc70,
    0x289b7ec6, 0xeaa127fa, 0xd4ef3085, 0x04881d05,
    0xd9d4d039, 0xe6db99e5, 0x1fa27cf8, 0xc4ac5665,
    0xf4292244, 0x432aff97, 0xab9423a7, 0xfc93a039,
    0x655b59c3, 0x8f0ccc92, 0xffeff47d, 0x85845dd1,
    0x6fa87e4f, 0xfe2ce6e0, 0xa3014314, 0x4e0811a1,
    0xf7537e82, 0xbd3af235, 0x2ad7d2bb, 0xeb86d391
  };

  uint8_t block[64] = {};
  cuda::std::memcpy(block, msg, len);
  block[len] = 0x80;
  uint64_t bit_len = (uint64_t)len * 8;
  cuda::std::memcpy(block + 56, &bit_len, sizeof(bit_len));

  uint32_t M[16];
  cuda::std::memcpy(M, block, sizeof(M));

  uint32_t a0 = 0x67452301, b0 = 0xefcdab89, c0 = 0x98badcfe, d0 = 0x10325476;
  uint32_t A = a0, B = b0, C = c0, D = d0;
  for (uint32_t i = 0; i < 64; i++) {
    uint32_t F, g;
    if (i < 16) {
      F = (B & C) | (~B & D);
      g = i;
    } else if (i < 32) {
      F = (D & B) | (~D & C);
      g = (5 * i + 1) % 16;
    } else if (i < 48) {
      F = B ^ C ^ D;
      g = (3 * i + 5) % 16;
    } else {
      F = C ^ (B | ~D);
      g = (7 * i) % 16;
    }
    F = F + A + K[i] + M[g];
    A = D;
    D = C;
    C = B;
    B = B + cuda::std::rotl(F, s[i]);
  }
  a0 += A;
  b0 += B;
  c0 += C;
  d0 += D;

  uint32_t out[4] = {a0, b0, c0, d0};
  cuda::std::memcpy(digest, out, sizeof(out));
}

Writing md5.cuh


In [ ]:
%%writefile hash_kernel.cuh
#pragma once
#include <cstdint>
#include <cuda/std/algorithm>
#include <cuda/std/array>
#include <cuda/std/charconv>
#include "get_thread.cuh"
#include "md5.cuh"

__global__
static void hash_kernel(uint32_t* value, bool* found, const cuda::std::array<uint8_t, md5_digest_size> target_digest) {
  if (*found) {
    return;
  }
  
  uint32_t guess = get_thread();

  constexpr int max_digits = 10;
  char char_arr[max_digits];
  auto uint8_arr = reinterpret_cast<const uint8_t*>(char_arr);
  auto [format_end, format_err] = cuda::std::to_chars(char_arr, char_arr + max_digits, guess);

  uint8_t value_digest[md5_digest_size];
  md5(value_digest, uint8_arr, format_end - char_arr);

  if (!cuda::std::equal(value_digest, value_digest + md5_digest_size, target_digest.begin())) {
    return;
  }

  *value = guess;
  *found = true;
}

Writing hash_kernel.cuh


In [384]:
%%writefile run_hash.cuh
#pragma once
#include <cstdint>
#include <cuda/std/algorithm>
#include <cuda/std/array>
#include <cuda_runtime.h>
#include "hash_kernel.cuh"

static inline cudaError_t run_hash(uint32_t* value_host, bool* found_host, const uint8_t* target) {
  constexpr uint32_t device_id = 0;

  auto set_device_err = cudaSetDevice(device_id);
  if (set_device_err != cudaSuccess) {
    return set_device_err;
  }

  cudaDeviceProp prop{};
  auto prop_err = cudaGetDeviceProperties(&prop, device_id);
  if (prop_err != cudaSuccess) {
    return prop_err;
  }

  uint32_t block_size = prop.maxThreadsPerBlock;
  uint32_t grid_size = UINT32_MAX / block_size + 1;

  cuda::std::array<uint8_t, md5_digest_size> target_host{};
  cuda::std::copy_n(target, md5_digest_size, target_host.begin());

  uint32_t* value_device{};
  bool* found_device{};

  auto value_malloc_err = cudaMallocManaged(&value_device, sizeof(uint32_t));
  if (value_malloc_err != cudaSuccess) {
    return value_malloc_err;
  }

  auto found_malloc_err = cudaMallocManaged(&found_device, sizeof(bool));
  if (found_malloc_err != cudaSuccess) {
    return found_malloc_err;
  }

  *found_device = false;

  hash_kernel<<<grid_size, block_size>>>(value_device, found_device, target_host);
  auto kernel_err = cudaGetLastError();
  if (kernel_err != cudaSuccess) {
    return kernel_err;
  }

  auto sync_err = cudaDeviceSynchronize();
  if (sync_err != cudaSuccess) {
    return sync_err;
  }

  *value_host = *value_device;
  *found_host = *found_device;

  cudaFree(value_device);
  cudaFree(found_device);

  return cudaSuccess;
}

Writing run_hash.cuh


In [385]:
%%writefile hash_lib.cu
#include <cstdint>
#include <cuda_runtime.h>
#include "run_hash.cuh"

extern "C" {
  cudaError_t c_run_hash(uint32_t* value, bool* found, const uint8_t* target) {
    return run_hash(value, found, target);
  }

  const char* c_get_error_string(cudaError_t err) {
    return cudaGetErrorString(err);
  }
}

Writing hash_lib.cu


In [386]:
%%bash
nvcc -std=c++23 -arch=native -O3 --compiler-options '-fPIC' -shared hash_lib.cu -o hash_lib.so

In [387]:
%%writefile hash.py
import sys
from ctypes import (
    CDLL,
    POINTER,
    byref,
    c_bool,
    c_char_p,
    c_int,
    c_uint8,
    c_uint32,
)
from pathlib import Path

try:
    lib = CDLL(str(Path(__file__).parent / 'hash_lib.so'))
except OSError as e:
    raise RuntimeError("Failed to load hash_lib.so") from e

lib.c_run_hash.argtypes = [POINTER(c_uint32), POINTER(c_bool), POINTER(c_uint8)]
lib.c_run_hash.restype  = c_int

lib.c_get_error_string.argtypes = [c_int]
lib.c_get_error_string.restype  = c_char_p

def get_error_string(err: int) -> str:
    return lib.c_get_error_string(err).decode('utf-8')

def parse_hash(text: str) -> bytes:
    if len(text) != 32:
        raise ValueError(f"Invalid hash: expected 32 hex characters, got {len(text)}")
    try:
        return bytes.fromhex(text)
    except ValueError:
        raise ValueError(f"Invalid hash: {text} is not hexadecimal")

def run_hash(target: bytes) -> int | None:
    buf = (c_uint8 * 16)(*target)
    value = c_uint32()
    found = c_bool()
    if (err := lib.c_run_hash(byref(value), byref(found), buf)) != 0:
        raise RuntimeError(f"Hash failed: {get_error_string(err)}")
    if not found.value:
        return None
    return value.value

if __name__ == '__main__':
    if len(sys.argv) != 2:
        print(f"Usage: {sys.argv[0]} <hash>", file=sys.stderr)
        sys.exit(1)

    target = parse_hash(sys.argv[1])
    value = run_hash(target)
    if value is None:
        print("No match found", file=sys.stderr)
        sys.exit(1)

    print(value)

Writing hash.py


In [388]:
%%bash
value=$SRANDOM
value_hash=$(echo -n $value | md5sum | awk '{print $1}')
guess=$(python3 hash.py $value_hash)

echo "Expected: $value"
echo "Got:      $guess"

if [ "$guess" -eq "$value" ]; then
  echo "Correct!"
else
  echo "Incorrect!"
fi

Expected: 871776877
Got:      871776877
Correct!
